# Урок 4. Сумматор стенограмм звонков

В этом уроке мы собираемся написать сложную подсказку для обычного сценария использования клиента: подведение итогов.  В частности, мы приведем длинные стенограммы звонков в службу поддержки клиентов.  Наша цель — обобщить обращения в службу поддержки клиентов для получения показателей поддержки клиентов.  Нам нужны сводки полных обращений в службу поддержки клиентов, чтобы оценить эффективность нашей команды поддержки клиентов.  Это означает, что мы исключим звонки, у которых есть проблемы с подключением, языковые барьеры и другие проблемы, которые мешают эффективному обобщению.

Давайте представим, что мы работаем в Acme Corporation, компании, которая продает устройства для умного дома. Компания ежедневно обрабатывает сотни звонков в службу поддержки клиентов, и ей нужен способ быстро превратить эти разговоры в **полезные структурированные данные**. 

Некоторые важные соображения включают в себя:
* Звонки могут быть короткими и приятными или длинными и сложными.
* Клиенты могут звонить по любому поводу: от простой проблемы с подключением Wi-Fi до сложной неисправности системы.
* Нам нужны наши сводки в определенном формате, чтобы их можно было легко проанализировать позже.
* Мы должны быть осторожны и не включать в наши сводки какую-либо личную информацию о клиентах.

Чтобы помочь нам, мы будем следовать лучшим практикам, которые мы описали ранее:
* Используйте системную подсказку, чтобы подготовить почву.
* Структурируйте приглашение для оптимальной производительности.
* Дайте четкие инструкции и определите желаемый результат.
* Используйте теги XML для организации информации.
* Обработка особых случаев и крайних сценариев.
* Приведите примеры для модели.

---

## Понимание данных

Теперь, когда мы понимаем нашу задачу, давайте посмотрим на данные, с которыми мы будем работать. В этом уроке мы будем использовать различные симулированные стенограммы звонков в службу поддержки клиентов из службы поддержки устройств умного дома корпорации Acme. Эти расшифровки помогут нам создать надежную подсказку, способную обрабатывать различные сценарии.

Давайте рассмотрим некоторые типы расшифровок звонков, с которыми мы можем столкнуться:

Краткая и простая расшифровка:

In [42]:
call1 = """
Agent: Thank you for calling Acme Smart Home Support. This is Alex. How can I help you?
Customer: Hi, I can't turn on my smart light bulb.
Agent: I see. Have you tried resetting the bulb?
Customer: Oh, no. How do I do that?
Agent: Just turn the power off for 5 seconds, then back on. It should reset.
Customer: Ok, I'll try that. Thanks!
Agent: You're welcome. Call us back if you need further assistance.
"""

Стенограмма средней длины с окончательным решением:

In [43]:
call2 = """
Agent: Acme Smart Home Support, this is Jamie. How may I assist you today?
Customer: Hi Jamie, my Acme SmartTherm isn't maintaining the temperature I set. It's set to 72 but the house is much warmer.
Agent: I'm sorry to hear that. Let's troubleshoot. Is your SmartTherm connected to Wi-Fi?
Customer: Yes, the Wi-Fi symbol is showing on the display.
Agent: Great. Let's recalibrate your SmartTherm. Press and hold the menu button for 5 seconds.
Customer: Okay, done. A new menu came up.
Agent: Perfect. Navigate to "Calibration" and press select. Adjust the temperature to match your room thermometer.
Customer: Alright, I've set it to 79 degrees to match.
Agent: Great. Press select to confirm. It will recalibrate, which may take a few minutes. Check back in an hour to see if it's fixed.
Customer: Okay, I'll do that. Thank you for your help, Jamie.
Agent: You're welcome! Is there anything else I can assist you with today?
Customer: No, that's all. Thanks again.
Agent: Thank you for choosing Acme Smart Home. Have a great day!
"""

Более длинный звонок без разрешения:

In [44]:
call3 = """
Agent: Thank you for contacting Acme Smart Home Support. This is Sarah. How can I help you today?
Customer: Hi Sarah, I'm having trouble with my Acme SecureHome system. The alarm keeps going off randomly.
Agent: I'm sorry to hear that. Can you tell me when this started happening?
Customer: It started about two days ago. It's gone off three times now, always in the middle of the night.
Agent: I see. Are there any error messages on the control panel when this happens?
Customer: No, I didn't notice any. But I was pretty groggy each time.
Agent: Understood. Let's check a few things. First, can you confirm that all your doors and windows are closing properly?
Customer: Yes, I've checked all of them. They're fine.
Agent: Okay. Next, let's check the battery in your control panel. Can you tell me if the low battery indicator is on?
Customer: Give me a moment... No, the battery indicator looks normal.
Agent: Alright. It's possible that one of your sensors is malfunctioning. I'd like to run a diagnostic, but I'll need to transfer you to our technical team for that. Is that okay?
Customer: Yes, that's fine. I just want this fixed. It's really disruptive.
Agent: I completely understand. I'm going to transfer you now. They'll be able to run a full system diagnostic and hopefully resolve the issue for you.
Customer: Okay, thank you.
Agent: You're welcome. Thank you for your patience, and I hope you have a great rest of your day.
"""

Эти примеры демонстрируют разнообразие вызовов и особенностей, которые нам необходимо обработать:
* Звонки имеют совершенно разную продолжительность.
* Звонки касаются различных вопросов поддержки (простые исправления, неисправности устройства, сложные проблемы).
* Некоторые звонки заканчиваются разрешением, а другие остаются неразрешенными.
* Некоторые звонки требуют дальнейшего контроля.

При создании приглашения нам необходимо убедиться, что оно может эффективно суммировать все эти типы вызовов, извлекая ключевую информацию и представляя ее в последовательном, структурированном формате.
В следующем разделе мы начнем шаг за шагом создавать нашу подсказку для обработки этого разнообразного диапазона расшифровок вызовов. 

---

## Простая версия приглашения
Теперь, когда мы понимаем нашу задачу и тип данных, с которыми работаем, давайте начнем создавать подсказку. Мы начнем с базовой версии и постепенно усовершенствуем ее, чтобы справиться со сложностями нашей задачи суммирования вызовов.

Начнем с очень простой подсказки, в которой описывается основная задача:

In [45]:
prompt = """
Summarize the following customer service call transcript. Focus on the main issue, how it was resolved, and any required follow-up.

{transcript}
"""

Эта базовая подсказка дает Клоду общее представление о том, чего мы хотим, но имеет несколько ограничений:

* Не указан желаемый формат вывода, что может привести к противоречивым сводкам.
* Он не содержит рекомендаций о том, как действовать в различных сценариях (например, нерешенных проблемах или недостаточной информации).
* Он не устанавливает никаких ограничений по длине или содержанию, что может привести к созданию слишком длинных или подробных изложений.
* Он не требует от Клода не указывать личную информацию, что может привести к проблемам с конфиденциальностью.

С учетом вышесказанного, давайте протестируем его, чтобы понять, как он работает:

In [46]:
from anthropic import Anthropic
from dotenv import load_dotenv

load_dotenv()
client = Anthropic()

def summarize_call(transcript):
    final_prompt = prompt.format(transcript=transcript)
    # Make the API call
    response = client.messages.create(
        model="claude-3-sonnet-20240229",
        max_tokens=4096,
        messages=[
            {"role": "user", "content": final_prompt}
        ]
    )
    print(response.content[0].text)

In [47]:
summarize_call(call1)

Here is a summary of the customer service call transcript:

Main Issue:
The customer was unable to turn on their Acme smart light bulb.

Resolution:
The service agent instructed the customer to reset the bulb by turning the power off for 5 seconds and then back on. This should reset the bulb and allow it to turn on.

Follow-Up:
The agent told the customer to call back if they continued to have issues after trying the reset procedure. No other follow-up was mentioned.


In [48]:
summarize_call(call2)

Summary:

Main Issue: The customer's Acme SmartTherm thermostat was not maintaining the set temperature of 72°F, and the house was much warmer.

Resolution: The agent guided the customer through the process of recalibrating the SmartTherm thermostat. This involved accessing the "Calibration" menu, adjusting the temperature to match the customer's room thermometer (79°F in this case), and confirming the new setting. The recalibration process may take a few minutes to complete.

Follow-up Required: The customer was advised to check the thermostat in an hour to see if the issue was resolved after the recalibration process completed.


In [49]:
summarize_call(call3)

Here is a summary of the customer service call transcript:

Main Issue:
The customer was having an issue with their Acme SecureHome alarm system going off randomly in the middle of the night, even though all doors and windows were closed properly.

How It Was Resolved:
The customer service agent first had the customer check for any error messages on the control panel and confirm that the battery was not low. When those basic troubleshooting steps did not reveal the issue, the agent determined that one of the sensors may be malfunctioning and needed to transfer the customer to the technical support team for a full system diagnostic.

Required Follow-Up:
The technical support team needs to run a diagnostic on the customer's SecureHome system to identify which sensor(s) may be causing the false alarms and then repair or replace those components. The customer should be contacted again once the diagnostic is complete and the repair/replacement has been performed to ensure the random alarms 

Как видите, хотя Клод и предоставляет резюме, оно не в том формате, который можно было бы легко анализировать систематически. Краткое изложение может быть слишком длинным или слишком коротким и не всегда охватывать все интересующие нас моменты.


На следующих шагах мы начнем добавлять больше структуры и рекомендаций в наше приглашение для устранения этих ограничений. Посмотрим, как каждое дополнение улучшит качество и последовательность резюме Клода.

Помните, что оперативное проектирование — это итеративный процесс. Начинаем с простого и постепенно дорабатываем нашу подсказку.

---

## Добавляем системное приглашение

Проще всего начать с системной подсказки, которая задает общий контекст и роль Клода, помогая управлять его поведением на протяжении всего взаимодействия.

Начнем с этой системной подсказки:

In [50]:
system = """
You are an expert customer service analyst, skilled at extracting key information from call transcripts and summarizing them in a structured format.
Your task is to analyze customer service call transcripts and generate concise, accurate summaries while maintaining a professional tone.
"""

--- 

## Структурируем наше основное приглашение

Далее мы начнем писать основную подсказку.  Мы будем опираться на некоторые из этих подсказок:

- Поместите длинные документы (наши стенограммы) вверху.
- Добавьте подробные инструкции и требования к выходному формату.
- Внедрение тегов XML для структурирования приглашения и вывода.
- Дайте Клоду возможность «поразмыслить вслух». 

Поскольку эта подсказка может оказаться довольно длинной, мы напишем отдельные фрагменты по отдельности, а затем объединим их вместе.

### Входные данные
При работе с большими языковыми моделями, такими как Claude, очень важно помещать длинные документы, например расшифровки разговоров, в начало подсказки. Это гарантирует, что Клод получит весь необходимый контекст перед получением конкретных инструкций. Нам также следует использовать теги XML для идентификации транскрипта в приглашении:

In [51]:
prompt_pt1 = """
Analyze the following customer service call transcript and generate a JSON summary of the interaction:

<transcript>
[INSERT CALL TRANSCRIPT HERE]
</transcript>
"""

### Инструкции и формат вывода

Прежде чем идти дальше, давайте хорошенько подумаем о том, как может выглядеть хороший структурированный формат вывода.  Чтобы облегчить нам жизнь при анализе результатов, зачастую проще всего запросить у Клода ответ в формате JSON.  Как в этом случае должен выглядеть хороший JSON?

Как минимум, наш вывод JSON должен включать следующее:
- Статус того, было ли у Клода достаточно информации для создания сводки.  Мы еще вернемся к этому.  На данный момент мы предполагаем, что все сводки имеют статус «ЗАВЕРШЕНО», что означает, что Клод может создать сводку.
- Краткое описание проблемы клиента
- Если звонок требует дополнительных действий
- Подробная информация о любых последующих действиях, если потребуется (перезвон клиенту и т. д.)
- Как решился вопрос
- Список двусмысленностей или неясных моментов в разговоре.

Вот предлагаемый пример структуры JSON:```json
{
  "summary": {
    "customerIssue": "Brief description of the main problem or reason for the call",
    "resolution": "How the issue was addressed or resolved, if applicable",
    "followUpRequired": true/false,
    "followUpDetails": "Description of any necessary follow-up actions, or null if none required"
  },
  "status": "COMPLETE",
  "ambiguities": ["List of any unclear or vague points in the conversation, or an empty array if none"]
}
```Давайте создадим новую часть нашей подсказки, включающую конкретные инструкции, в том числе:
- Создайте резюме, сосредоточив внимание на основной проблеме, решении и любых необходимых последующих действиях.
– Сгенерируйте вывод JSON в соответствии с нашим особым стандартизированным форматом.
- Опустите конкретную информацию о клиенте в сводках.
- Делайте каждую часть резюме краткой.

Вот попытка предоставить инструкции вывода, включая наш специальный формат вывода JSON:

In [52]:
prompt_pt2 = """
Instructions:
1. Read the transcript carefully.
2. Analyze the transcript, focusing on the main issue, resolution, and any follow-up required.
3. Generate a JSON object summarizing the key aspects of the interaction according to the specified structure.

Important guidelines:
- Confidentiality: Omit all specific customer data like names, phone numbers, and email addresses.
- Character limit: Restrict each text field to a maximum of 100 characters.
- Maintain a professional tone in your summary.

Output format:
Generate a JSON object with the following structure:
<json>
{
  "summary": {
    "customerIssue": "Brief description of the main problem or reason for the call",
    "resolution": "How the issue was addressed or resolved, if applicable",
    "followUpRequired": true/false,
    "followUpDetails": "Description of any necessary follow-up actions, or null if none required"
  },
  "status": "COMPLETE",
  "ambiguities": ["List of any unclear or vague points in the conversation, or an empty array if none"]
}
</json>
"""

--- 

## Использование XML-тегов и предоставление Клоду места для размышлений
Далее мы применим еще две стратегии подсказок: предоставив Клоду возможность подумать и используя теги XML.
- Мы попросим Клода начать с вывода тегов `<thinking>`, содержащих его анализ.
- Затем мы попросим Claude вывести вывод JSON внутри <json>.

Вот заключительная часть нашего первого черновика подсказки:

In [53]:
prompt_pt3 = """
Before generating the JSON, please analyze the transcript in <thinking> tags. 
Include your identification of the main issue, resolution, follow-up requirements, and any ambiguities. 
Then, provide your JSON output in <json> tags.
"""

Прося Клода поместить свой анализ в теги «<thinking>», мы предлагаем ему разбить мыслительный процесс перед формулированием окончательного вывода в формате JSON. Это побуждает к более тщательному и структурированному подходу к анализу стенограммы.
Раздел «<thinking>» позволяет нам (и, возможно, другим рецензентам или системам) видеть процесс рассуждений Клода. Эта прозрачность может иметь решающее значение для целей отладки и обеспечения качества.


Отделяя анализ (`<thinking`>) от структурированного вывода (`<json>`), мы создаем четкое различие между интерпретацией Клодом стенограммы и ее форматированным резюме. Это может быть полезно в тех случаях, когда нам может потребоваться просмотреть анализ отдельно от вывода JSON, но также, изолируя содержимое JSON внутри тегов <json>, мы упрощаем анализ окончательного ответа и получение JSON, с которым мы хотим работать.

--- 

## Тестируем обновленную подсказку

Вот полная версия подсказки, созданная путем объединения отдельных частей подсказки, которые мы написали до сих пор:

In [56]:
system = """
You are an expert customer service analyst, skilled at extracting key information from call transcripts and summarizing them in a structured format.
Your task is to analyze customer service call transcripts and generate concise, accurate summaries while maintaining a professional tone.
"""

prompt = """
Analyze the following customer service call transcript and generate a JSON summary of the interaction:

<transcript>
[INSERT CALL TRANSCRIPT HERE]
</transcript>

Instructions:
1. Read the transcript carefully.
2. Analyze the transcript, focusing on the main issue, resolution, and any follow-up required.
3. Generate a JSON object summarizing the key aspects of the interaction according to the specified structure.

Important guidelines:
- Confidentiality: Omit all specific customer data like names, phone numbers, and email addresses.
- Character limit: Restrict each text field to a maximum of 100 characters.
- Maintain a professional tone in your summary.

Output format:
Generate a JSON object with the following structure:
<json>
{
  "summary": {
    "customerIssue": "Brief description of the main problem or reason for the call",
    "resolution": "How the issue was addressed or resolved, if applicable",
    "followUpRequired": true/false,
    "followUpDetails": "Description of any necessary follow-up actions, or null if none required"
  },
  "status": "COMPLETE",
  "ambiguities": ["List of any unclear or vague points in the conversation, or an empty array if none"]
}
</json>

Before generating the JSON, please analyze the transcript in <thinking> tags. 
Include your identification of the main issue, resolution, follow-up requirements, and any ambiguities. 
Then, provide your JSON output in <json> tags.
"""

Вот функция, которую мы можем использовать для проверки нашего приглашения:

In [57]:
def summarize_call_with_improved_prompt(transcript):
    final_prompt = prompt.replace("[INSERT CALL TRANSCRIPT HERE]", transcript)
    # Make the API call
    response = client.messages.create(
        model="claude-3-sonnet-20240229",
        system=system,
        max_tokens=4096,
        messages=[
            {"role": "user", "content": final_prompt}
        ]
    )
    print(response.content[0].text)

Давайте проверим приглашение, используя некоторые из ранее определенных нами расшифровок вызовов:

In [58]:
summarize_call_with_improved_prompt(call1)

<thinking>
From the transcript, the main issue appears to be that the customer could not turn on their smart light bulb. The resolution provided by the agent was to reset the bulb by turning the power off for 5 seconds and then back on.

The agent did offer for the customer to call back if they needed further assistance, indicating potential follow-up may be required if the reset did not resolve the issue. However, no specific follow-up details were provided.

There do not seem to be any significant ambiguities in the conversation.
</thinking>

<json>
{
  "summary": {
    "customerIssue": "Unable to turn on smart light bulb",
    "resolution": "Agent instructed customer to reset the bulb by turning power off for 5 seconds, then back on",
    "followUpRequired": true,
    "followUpDetails": "Customer was advised to call back if the reset did not resolve the issue"
  },
  "status": "COMPLETE",
  "ambiguities": []
}
</json>


In [59]:
summarize_call_with_improved_prompt(call2)

<thinking>
Main issue: The customer's Acme SmartTherm thermostat is not maintaining the set temperature of 72°F, and the house is much warmer.

Resolution: The agent guided the customer through recalibrating the SmartTherm thermostat by:
1. Having the customer press and hold the menu button for 5 seconds.
2. Navigating to the "Calibration" menu and selecting it.
3. Adjusting the temperature to match the customer's room thermometer reading of 79°F.
4. Confirming the new calibration setting.

Follow-up required: Yes, the agent instructed the customer to check back in an hour to see if the recalibration resolved the temperature issue.

Ambiguities: None
</thinking>

<json>
{
  "summary": {
    "customerIssue": "Thermostat not maintaining set temperature, causing house to be much warmer.",
    "resolution": "Agent guided customer through recalibrating the thermostat to match room temperature.",
    "followUpRequired": true,
    "followUpDetails": "Customer to check back in an hour to see i

In [60]:
summarize_call_with_improved_prompt(call3)

<thinking>
Main issue: The customer's Acme SecureHome system alarm is going off randomly in the middle of the night, even though doors and windows are closed properly.

Resolution: The agent suggests running a diagnostic on the system to identify potential sensor malfunctions. The customer is transferred to the technical team to perform the diagnostic and resolve the issue.

Follow-up required: Yes, the technical team needs to follow up with the customer to diagnose and fix the alarm system problem.

Ambiguities: None identified in the conversation.
</thinking>

<json>
{
  "summary": {
    "customerIssue": "Customer's home security alarm system is going off randomly at night without apparent cause.",
    "resolution": "Agent suggests running a diagnostic to check for sensor malfunction and transfers customer to technical team.",
    "followUpRequired": true,
    "followUpDetails": "Technical team to diagnose and resolve the issue with the customer's alarm system."
  },
  "status": "COM

Все эти ответы выглядят великолепно! Давайте попробуем другую расшифровку вызова, в которой есть некоторая двусмысленность, чтобы увидеть, включает ли результат JSON эти двусмысленности:

In [64]:
ambiguous_call = """
Agent: Thank you for calling Acme Smart Home Support. This is Alex. How may I assist you today?
Customer: Hi Alex, I'm having an issue with my SmartLock. It's not working properly.
Agent: I'm sorry to hear that. Can you tell me more about what's happening with your SmartLock?
Customer: Well, sometimes it doesn't lock when I leave the house. I think it might be related to my phone, but I'm not sure.
Agent: I see. When you say it doesn't lock, do you mean it doesn't respond to the auto-lock feature, or are you trying to lock it manually through the app?
Customer: Uh, both, I think. Sometimes one works, sometimes the other. It's inconsistent.
Agent: Okay. And you mentioned it might be related to your phone. Have you noticed any pattern, like it works better when you're closer to the door?
Customer: Maybe? I haven't really paid attention to that.
Agent: Alright. Let's try to troubleshoot this. First, can you tell me what model of SmartLock you have?
Customer: I'm not sure. I bought it about six months ago, if that helps.
Agent: That's okay. Can you see a model number on the lock itself?
Customer: I'd have to go check. Can we just assume it's the latest model?
Agent: Well, knowing the exact model would help us troubleshoot more effectively. But let's continue with what we know. Have you tried resetting the lock recently?
Customer: I think so. Or maybe that was my SmartTherm. I've been having issues with that too.
Agent: I see. It sounds like we might need to do a full diagnostic on your SmartLock. Would you be comfortable if I walked you through that process now?
Customer: Actually, I have to run to an appointment. Can I call back later?
Agent: Of course. Before you go, is there a good contact number where our technical team can reach you for a more in-depth troubleshooting session?
Customer: Sure, you can reach me at 555... oh wait, that's my old number. Let me check my new one... You know what, I'll just call back when I have more time.
Agent: I understand. We're here 24/7 when you're ready to troubleshoot. Is there anything else I can help with before you go?
Customer: No, that's it. Thanks.
Agent: You're welcome. Thank you for choosing Acme Smart Home. Have a great day!
"""

summarize_call_with_improved_prompt(ambiguous_call)

<thinking>
Main Issue: The customer is experiencing issues with their Acme SmartLock not consistently locking automatically or manually through the app.

Resolution: The agent attempted to troubleshoot by asking for the specific SmartLock model and suggesting a reset, but the customer had to leave before completing the troubleshooting process.

Follow-Up Required: Yes, the customer needs to call back to complete a full diagnostic and troubleshooting session with the technical team.

Ambiguities:
- The customer was unsure if the issue was related to their phone or not, suggesting a potential connectivity problem.
- The customer mentioned having issues with another Acme product (SmartTherm), but it's unclear if those issues are related to the SmartLock problem.
- The customer's contact number was not clearly provided, which could make follow-up more difficult.
</thinking>

<json>
{
  "summary": {
    "customerIssue": "SmartLock not consistently locking automatically or manually through t

Большой! Кажется, все работает так, как задумано

--- 

## Краевые случаи

До сих пор все расшифровки звонков, которые мы пробовали, представляли собой относительно простые звонки в службу поддержки клиентов. В реальном мире мы также ожидаем встретить стенограммы, которые, возможно, мы не хотим резюмировать, в том числе: 

- Звонки при проблемах со связью
- Звонки с языковым барьером
- Звонки с искаженными стенограммами
- Звонки иррациональным или расстроенным клиентам.

Помните, наша цель — обобщить эти звонки, чтобы оценить эффективность предлагаемого нами обслуживания клиентов.  Если мы включим эти вызовы в крайних случаях в сводку, мы, скорее всего, получим искаженные результаты.

Давайте посмотрим, что произойдет в некоторых из этих крайних случаев с помощью нашего текущего приглашения.  Ниже мы определили некоторые новые расшифровки вызовов:

In [65]:
wrong_number_call = """
Agent: Acme Smart Home Support, Lisa speaking. How can I help you?
Customer: Is this tech support?
Agent: Yes, this is technical support for Acme Smart Home devices. What can I help you with?
Customer: Sorry, wrong number.
Agent: No problem. Have a nice day.
"""

incomplete_call = """
Agent: Acme Smart Home Support, this is Sarah. How can I assist you today?
Customer: The thing isn't working.
Agent: I'm sorry to hear that. Could you please specify which device you're having trouble with?
Customer: You know, the usual one. Gotta go, bye.
Agent: Wait, I need more infor... [call disconnected]
"""

garbled_call = """
Agent: Thank you for calling Acme Smart Home Support. This is Alex. How may I assist you today?
Customer: [garbled voice]
Agent: Hello? Are you there?
"""

language_barrier_call = """
Agent: Acme Smart Home Support, Sarah speaking. How can I help you today?
Customer: [Speaking in Spanish]
Agent: I apologize, but I don't speak Spanish. Do you speak English?
Customer: [Continues Spanish]
Agent: One moment please, I'll try to get a translator on the line...
"""


Давайте пропустим эти расшифровки крайних случаев через нашу подсказку и посмотрим, какие результаты мы получим:

In [66]:
summarize_call_with_improved_prompt(wrong_number_call)

<thinking>
Issue: The customer appears to have dialed the wrong number for technical support.
Resolution: Since it was a wrong number, there was no issue to resolve. The agent politely concluded the call.
Follow-up: No follow-up is required since it was a misdialed call.
Ambiguities: There are no apparent ambiguities in this brief conversation.
</thinking>

<json>
{
  "summary": {
    "customerIssue": "The customer dialed the wrong number for technical support",
    "resolution": "The agent concluded the call politely since it was a misdialed number",
    "followUpRequired": false,
    "followUpDetails": null
  },
  "status": "COMPLETE",
  "ambiguities": []
}
</json>


In [67]:
summarize_call_with_improved_prompt(incomplete_call)

<thinking>
In this transcript, the main issue is unclear as the customer does not provide sufficient details about the device they are having trouble with. The agent attempts to clarify the issue, but the customer disconnects the call before providing more information.

There is no resolution since the issue is not clearly identified. However, follow-up is required to gather more details from the customer about the specific device and the problem they are experiencing.

The key ambiguity is the lack of clarity about the device and the nature of the problem. The customer's vague statements ("the thing isn't working" and "the usual one") do not provide enough information for the agent to diagnose or resolve the issue.
</thinking>

<json>
{
  "summary": {
    "customerIssue": "Customer reported an unspecified device was not working but did not provide further details.",
    "resolution": "No resolution was possible due to lack of information from the customer.",
    "followUpRequired": tr

In [68]:
summarize_call_with_improved_prompt(garbled_call)

<thinking>
From the transcript, it appears the customer called Acme Smart Home Support, but their voice was garbled, and the agent could not understand them. With this limited information, it is unclear what the main issue or reason for the call was. There was no resolution provided, and it is ambiguous whether follow-up is required since the issue itself is unknown. The key ambiguity is the lack of clear communication from the customer, preventing the agent from understanding the problem.
</thinking>

<json>
{
  "summary": {
    "customerIssue": "Unclear due to garbled voice from the customer",
    "resolution": "No resolution provided since the issue could not be understood",
    "followUpRequired": true,
    "followUpDetails": "Agent should try to reconnect with the customer for clearer communication"
  },
  "status": "COMPLETE",
  "ambiguities": ["The customer's voice was garbled, preventing understanding of the issue"]
}
</json>


In [69]:
summarize_call_with_improved_prompt(language_barrier_call)

<thinking>
Main issue: The customer called and spoke in Spanish, but the agent could not understand Spanish.
Resolution: The agent tried to get a translator on the line to resolve the language barrier.
Follow-up required: Yes, the agent needs to connect with a Spanish translator to assist the customer.
Ambiguities: It is unclear why the customer called, as the reason for their call is not stated in the transcript.
</thinking>

<json>
{
  "summary": {
    "customerIssue": "Customer spoke in a language the agent did not understand (Spanish).",
    "resolution": "Agent attempted to get a translator to resolve the language barrier.",
    "followUpRequired": true,
    "followUpDetails": "Agent needs to connect the customer with a Spanish translator."
  },
  "status": "COMPLETE",
  "ambiguities": ["Reason for the customer's call is not stated in the transcript."]
}
</json>


К сожалению, мы получаем полные сводки по этим крайним случаям стенограмм.  Вот некоторые проблемные части ответов: 

> "customerIssue": "Клиент говорил на языке, который агент не понимал (испанский)".

> "customerIssue": "Неясно из-за искажения голоса клиента"

> "customerIssue": "Клиент набрал неправильный номер службы технической поддержки" 

Помните, что наша цель — обобщить наши звонки в службу поддержки клиентов, чтобы получить представление о том, насколько эффективна наша команда обслуживания клиентов.  Эти расшифровки крайних случаев приводят к созданию полных сводок, что может вызвать проблемы при анализе всех сводок.  Нам нужно будет принять решение о стратегии обработки этих звонков.

--- 

## Дальнейшие быстрые улучшения

Как мы видели ранее, наша подсказка в настоящее время генерирует полные сводки для расшифровок крайних случаев.  Мы хотим изменить это поведение.  У нас есть несколько вариантов обработки этих крайних случаев:

- Пометьте их каким-либо образом, чтобы указать, что они не поддаются обобщению, что позволяет позднее просмотреть их человеком.
- Классифицируйте их отдельно (например, «технические трудности», «языковой барьер» и т. д.).

Для простоты мы решили пометить эти вызовы в крайнем случае, попросив модель вывести JSON, который выглядит следующим образом:```json
{
  "status": "INSUFFICIENT_DATA"
}
```Чтобы это заработало, нам нужно обновить нашу подсказку следующим образом:
- Добавьте инструкции, объясняющие желаемый вывод «НЕДОСТАТОЧНО_ДАННЫХ».
- Добавьте примеры, чтобы показать суммируемые и несуммируемые расшифровки вместе с соответствующими выходными данными JSON.

### Обновление нашей инструкции

Давайте напишем новую часть инструкций в подсказке, чтобы объяснить, когда модель должна выводить наш JSON «НЕДОСТАТОЧНО_ДАННЫХ».

In [70]:
# Just the new content.  We'll look at the entire prompt in a moment
new_instructions_addition = """
Insufficient data criteria:
   If either of these conditions are met:
   a) The transcript has fewer than 5 total exchanges, or
   b) The customer's issue is unclear
   c) The call is garbled, incomplete, or is hindered by a language barrier
   Then return ONLY the following JSON:
   {
     "status": "INSUFFICIENT_DATA"
   }
"""

### Добавление примеров

Как мы уже обсуждали ранее в этом курсе, почти всегда полезно добавлять примеры в подсказку.  В этом конкретном случае примеры помогут Клоду в целом понять, какие типы сводок нам нужны как для суммируемых, так и для несуммируемых расшифровок вызовов.

Вот набор примеров, которые мы могли бы включить в наше приглашение:

In [71]:
examples_for_prompt = """
<examples>
1. Complete interaction:
<transcript>
Agent: Thank you for calling Acme Smart Home Support. This is Alex. How may I assist you today?
Customer: Hi Alex, my Acme SmartTherm isn't maintaining the temperature I set. It's set to 72 but the house is much warmer.
Agent: I'm sorry to hear that. Let's troubleshoot. Is your SmartTherm connected to Wi-Fi?
Customer: Yes, the Wi-Fi symbol is showing on the display.
Agent: Great. Let's recalibrate your SmartTherm. Press and hold the menu button for 5 seconds.
Customer: Okay, done. A new menu came up.
Agent: Perfect. Navigate to "Calibration" and press select. Adjust the temperature to match your room thermometer.
Customer: Alright, I've set it to 79 degrees to match.
Agent: Great. Press select to confirm. It will recalibrate, which may take a few minutes. Check back in an hour to see if it's fixed.
Customer: Okay, I'll do that. Thank you for your help, Alex.
Agent: You're welcome! Is there anything else I can assist you with today?
Customer: No, that's all. Thanks again.
Agent: Thank you for choosing Acme Smart Home. Have a great day!
</transcript>

<thinking>
Main issue: SmartTherm not maintaining set temperature
Resolution: Guided customer through recalibration process
Follow-up: Not required, but customer should check effectiveness after an hour
Ambiguities: None identified
</thinking>

<json>
{
  "summary": {
    "customerIssue": "SmartTherm not maintaining set temperature, showing higher than set 72 degrees",
    "resolution": "Guided customer through SmartTherm recalibration process",
    "followUpRequired": false,
    "followUpDetails": null
  },
  "status": "COMPLETE",
  "ambiguities": []
}
</json>

2. Interaction requiring follow-up:
<transcript>
Agent: Acme Smart Home Support, this is Jamie. How can I help you?
Customer: Hi, I just installed my new Acme SmartCam, but I can't get it to connect to my Wi-Fi.
Agent: I'd be happy to help. Are you using the Acme Smart Home app?
Customer: Yes, I have the app on my phone.
Agent: Great. Make sure your phone is connected to the 2.4GHz Wi-Fi network, not the 5GHz one.
Customer: Oh, I'm on the 5GHz network. Should I switch?
Agent: Yes, please switch to the 2.4GHz network. The SmartCam only works with 2.4GHz.
Customer: Okay, done. Now what?
Agent: Open the app, select 'Add Device', choose 'SmartCam', and follow the on-screen instructions.
Customer: It's asking for a password now.
Agent: Enter your Wi-Fi password and it should connect.
Customer: It's still not working. I keep getting an error message.
Agent: I see. In that case, I'd like to escalate this to our technical team. They'll contact you within 24 hours.
Customer: Okay, that sounds good. Thank you for trying to help.
Agent: You're welcome. Is there anything else you need assistance with?
Customer: No, that's all for now. Thanks again.
Agent: Thank you for choosing Acme Smart Home. Have a great day!
</transcript>

<thinking>
Main issue: Customer unable to connect new SmartCam to Wi-Fi
Resolution: Initial troubleshooting unsuccessful, issue escalated to technical team
Follow-up: Required, technical team to contact customer within 24 hours
Ambiguities: Specific error message customer is receiving not mentioned
</thinking>

<json>
{
  "summary": {
    "customerIssue": "Unable to connect new SmartCam to Wi-Fi",
    "resolution": "Initial troubleshooting unsuccessful, issue escalated to technical team",
    "followUpRequired": true,
    "followUpDetails": "Technical team to contact customer within 24 hours for further assistance"
  },
  "status": "COMPLETE",
  "ambiguities": ["Specific error message customer is receiving not mentioned"]
}
</json>

3. Insufficient data:
<transcript>
Agent: Acme Smart Home Support, this is Sam. How may I assist you?
Customer: Hi, my smart lock isn't working.
Agent: I'm sorry to hear that. Can you tell me more about the issue?
Customer: It just doesn't work. I don't know what else to say.
Agent: Okay, when did you first notice the problem? And what model of Acme smart lock do you have?
Customer: I don't remember. Listen, I have to go. I'll call back later.
Agent: Alright, we're here 24/7 if you need further assistance. Have a good day.
</transcript>

<thinking>
This transcript has fewer than 5 exchanges and the customer's issue is unclear. The customer doesn't provide specific details about the problem with the smart lock or respond to the agent's questions. This interaction doesn't provide sufficient information for a complete summary.
</thinking>

<json>
{
  "status": "INSUFFICIENT_DATA"
}
</json>
</examples>
"""

Обратите внимание, что примеры охватывают три разные ситуации: 
* Полноценное взаимодействие, не требующее последующих действий
* Полное взаимодействие, требующее дальнейших действий и содержащее неясности.
* Взаимодействие, не поддающееся суммированию и содержащее недостаточно данных.

Предоставляя примеры Клоду, важно охватить различные пары ввода/вывода.

--- 

## Наша последняя подсказка

Давайте объединим нашу первоначальную подсказку с дополнениями, которые мы сделали в предыдущем разделе:
* инструкции по обработке звонков при недостаточном количестве данных
* набор примеров входов и выходов

Это новая полная подсказка:

In [75]:
system = """
You are an expert customer service analyst, skilled at extracting key information from call transcripts and summarizing them in a structured format.
Your task is to analyze customer service call transcripts and generate concise, accurate summaries while maintaining a professional tone.
"""

prompt = """
Analyze the following customer service call transcript and generate a JSON summary of the interaction:

<transcript>
[INSERT CALL TRANSCRIPT HERE]
</transcript>

Instructions:
<instructions>
1. Read the transcript carefully.
2. Analyze the transcript, focusing on the main issue, resolution, and any follow-up required.
3. Generate a JSON object summarizing the key aspects of the interaction according to the specified structure.

Important guidelines:
- Confidentiality: Omit all specific customer data like names, phone numbers, and email addresses.
- Character limit: Restrict each text field to a maximum of 100 characters.
- Maintain a professional tone in your summary.

Output format:
Generate a JSON object with the following structure:
<json>
{
  "summary": {
    "customerIssue": "Brief description of the main problem or reason for the call",
    "resolution": "How the issue was addressed or resolved, if applicable",
    "followUpRequired": true/false,
    "followUpDetails": "Description of any necessary follow-up actions, or null if none required"
  },
  "status": "COMPLETE",
  "ambiguities": ["List of any unclear or vague points in the conversation, or an empty array if none"]
}
</json>

Insufficient data criteria:
   If any of these conditions are met:
   a) The transcript has fewer than 5 total exchanges
   b) The customer's issue is unclear
   c) The call is garbled, incomplete, or is hindered by a language barrier
   Then return ONLY the following JSON:
   {
     "status": "INSUFFICIENT_DATA"
   }

Examples: 
<examples>
1. Complete interaction:
<transcript>
Agent: Thank you for calling Acme Smart Home Support. This is Alex. How may I assist you today?
Customer: Hi Alex, my Acme SmartTherm isn't maintaining the temperature I set. It's set to 72 but the house is much warmer.
Agent: I'm sorry to hear that. Let's troubleshoot. Is your SmartTherm connected to Wi-Fi?
Customer: Yes, the Wi-Fi symbol is showing on the display.
Agent: Great. Let's recalibrate your SmartTherm. Press and hold the menu button for 5 seconds.
Customer: Okay, done. A new menu came up.
Agent: Perfect. Navigate to "Calibration" and press select. Adjust the temperature to match your room thermometer.
Customer: Alright, I've set it to 79 degrees to match.
Agent: Great. Press select to confirm. It will recalibrate, which may take a few minutes. Check back in an hour to see if it's fixed.
Customer: Okay, I'll do that. Thank you for your help, Alex.
Agent: You're welcome! Is there anything else I can assist you with today?
Customer: No, that's all. Thanks again.
Agent: Thank you for choosing Acme Smart Home. Have a great day!
</transcript>

<thinking>
Main issue: SmartTherm not maintaining set temperature
Resolution: Guided customer through recalibration process
Follow-up: Not required, but customer should check effectiveness after an hour
Ambiguities: None identified
</thinking>

<json>
{
  "summary": {
    "customerIssue": "SmartTherm not maintaining set temperature, showing higher than set 72 degrees",
    "resolution": "Guided customer through SmartTherm recalibration process",
    "followUpRequired": false,
    "followUpDetails": null
  },
  "status": "COMPLETE",
  "ambiguities": []
}
</json>

2. Interaction requiring follow-up:
<transcript>
Agent: Acme Smart Home Support, this is Jamie. How can I help you?
Customer: Hi, I just installed my new Acme SmartCam, but I can't get it to connect to my Wi-Fi.
Agent: I'd be happy to help. Are you using the Acme Smart Home app?
Customer: Yes, I have the app on my phone.
Agent: Great. Make sure your phone is connected to the 2.4GHz Wi-Fi network, not the 5GHz one.
Customer: Oh, I'm on the 5GHz network. Should I switch?
Agent: Yes, please switch to the 2.4GHz network. The SmartCam only works with 2.4GHz.
Customer: Okay, done. Now what?
Agent: Open the app, select 'Add Device', choose 'SmartCam', and follow the on-screen instructions.
Customer: It's asking for a password now.
Agent: Enter your Wi-Fi password and it should connect.
Customer: It's still not working. I keep getting an error message.
Agent: I see. In that case, I'd like to escalate this to our technical team. They'll contact you within 24 hours.
Customer: Okay, that sounds good. Thank you for trying to help.
Agent: You're welcome. Is there anything else you need assistance with?
Customer: No, that's all for now. Thanks again.
Agent: Thank you for choosing Acme Smart Home. Have a great day!
</transcript>

<thinking>
Main issue: Customer unable to connect new SmartCam to Wi-Fi
Resolution: Initial troubleshooting unsuccessful, issue escalated to technical team
Follow-up: Required, technical team to contact customer within 24 hours
Ambiguities: Specific error message customer is receiving not mentioned
</thinking>

<json>
{
  "summary": {
    "customerIssue": "Unable to connect new SmartCam to Wi-Fi",
    "resolution": "Initial troubleshooting unsuccessful, issue escalated to technical team",
    "followUpRequired": true,
    "followUpDetails": "Technical team to contact customer within 24 hours for further assistance"
  },
  "status": "COMPLETE",
  "ambiguities": ["Specific error message customer is receiving not mentioned"]
}
</json>

3. Insufficient data:
<transcript>
Agent: Acme Smart Home Support, this is Sam. How may I assist you?
Customer: Hi, my smart lock isn't working.
Agent: I'm sorry to hear that. Can you tell me more about the issue?
Customer: It just doesn't work. I don't know what else to say.
Agent: Okay, when did you first notice the problem? And what model of Acme smart lock do you have?
Customer: I don't remember. Listen, I have to go. I'll call back later.
Agent: Alright, we're here 24/7 if you need further assistance. Have a good day.
</transcript>

<thinking>
This transcript has fewer than 5 exchanges and the customer's issue is unclear. The customer doesn't provide specific details about the problem with the smart lock or respond to the agent's questions. This interaction doesn't provide sufficient information for a complete summary.
</thinking>

<json>
{
  "status": "INSUFFICIENT_DATA"
}
</json>
</examples>
</instructions>

Before generating the JSON, please analyze the transcript in <thinking> tags. 
Include your identification of the main issue, resolution, follow-up requirements, and any ambiguities. 
Then, provide your JSON output in <json> tags.
"""

Приведенная выше подсказка довольно длинная, но вот ее общая структура:
- Системная подсказка задает контекст, роль и тон модели.
- Основная подсказка включает в себя следующее:
    - стенограмма разговора
    - комплект инструкций, содержащий:
        - общие инструкции
        - руководящие принципы 
        - требования к выходному формату
        - подробности об обработке вызовов в крайних случаях
        - примеры
    - подробности о тегах XML, которые будут использоваться в выходных данных

Вот краткое описание, которое поможет визуализировать ход выполнения приглашения:```txt
Analyze the following customer service call transcript and generate a JSON summary of the interaction:

<transcript>
[INSERT CALL TRANSCRIPT HERE]
</transcript>

<instructions>
- General instructions and guidelines
- Output JSON format description
- Insufficient data (edge-case) criteria
<examples>
varied example inputs and outputs
</examples>
</instructions>

Before generating the JSON, please analyze the transcript in <thinking> tags. 
Include your identification of the main issue, resolution, follow-up requirements, and any ambiguities. 
Then, provide your JSON output in <json> tags.

```

Давайте проверим последнее приглашение с помощью новой функции.  Обратите внимание, что эта функция извлекает сводное содержимое JSON внутри тегов <json>:

In [76]:
import re

def summarize_call_with_final_prompt(transcript):
    final_prompt = prompt.replace("[INSERT CALL TRANSCRIPT HERE]", transcript)
    # Make the API call
    response = client.messages.create(
        model="claude-3-sonnet-20240229",
        system=system,
        max_tokens=4096,
        messages=[
            {"role": "user", "content": final_prompt}
        ]
    )
    
    # Extract content between <json> tags
    json_content = re.search(r'<json>(.*?)</json>', response.content[0].text, re.DOTALL)
    
    if json_content:
        print(json_content.group(1).strip())
    else:
        print("No JSON content found in the response.")

Давайте проверим это на нескольких существующих переменных вызова:

In [77]:
summarize_call_with_final_prompt(call1)

{
  "summary": {
    "customerIssue": "Unable to turn on smart light bulb",
    "resolution": "Agent guided customer to reset the bulb by cycling power off and on",
    "followUpRequired": false,
    "followUpDetails": null
  },
  "status": "COMPLETE",
  "ambiguities": []
}


In [78]:
summarize_call_with_final_prompt(call3)

{
  "summary": {
    "customerIssue": "Acme SecureHome alarm system going off randomly multiple times at night without apparent cause",
    "resolution": "Initial troubleshooting steps taken, but issue unresolved. Customer transferred to technical team for diagnostics",
    "followUpRequired": true,
    "followUpDetails": "Technical team to diagnose and resolve issue with alarm system"
  },
  "status": "COMPLETE",
  "ambiguities": []
}


Давайте попробуем нашу расшифровку вызова, которая должна привести к выводу сводки с непустым массивом ambiguities:

In [79]:
summarize_call_with_final_prompt(ambiguous_call)

{
  "summary": {
    "customerIssue": "SmartLock not reliably locking automatically or through app, behavior is inconsistent",
    "resolution": "Troubleshooting attempted but incomplete due to lack of model details, customer had to leave",
    "followUpRequired": true,
    "followUpDetails": "Customer to call back for further troubleshooting of SmartLock issue when available"
  },
  "status": "COMPLETE",
  "ambiguities": [
    "Unclear if related SmartTherm issue mentioned",
    "SmartLock model not identified",
    "Customer's contact number not confirmed"
  ]
}


Теперь давайте попробуем некоторые из наших крайних подсказок, которые мы не хотим суммировать:

In [80]:
summarize_call_with_final_prompt(garbled_call)

{
  "status": "INSUFFICIENT_DATA"
}


In [82]:
summarize_call_with_final_prompt(language_barrier_call)

{
  "status": "INSUFFICIENT_DATA"
}


In [83]:
summarize_call_with_final_prompt(incomplete_call)

{
  "status": "INSUFFICIENT_DATA"
}


Большой! Мы получаем именно те результаты, которые хотим! Давайте попробуем продвинуться еще дальше:

In [84]:
summarize_call_with_final_prompt("blah blah blah")

{
  "status": "INSUFFICIENT_DATA"
}


In [85]:
summarize_call_with_final_prompt("")

{
  "status": "INSUFFICIENT_DATA"
}


Отлично, подсказка обрабатывает все крайние случаи!

---

## Заворачивать

В этом уроке мы рассмотрели процесс разработки сложной подсказки для обобщения стенограмм звонков в службу поддержки клиентов. Давайте подведем итоги использованных нами методов подсказки:

* Системная подсказка: мы использовали системную подсказку, чтобы установить общий контекст и роль Клода.
* Структурированный ввод: мы разместили расшифровку звонка в начале приглашения, используя теги XML.
* Четкие инструкции: мы предоставили подробные инструкции о том, на чем сосредоточиться и как структурировать результат.
* Форматирование вывода: мы указали структуру JSON для сводки, что обеспечивает единообразие и легкость анализа результатов.
* Обработка крайних случаев: мы добавили критерии для выявления вызовов с недостаточными данными.
* Примеры: мы включили различные примеры, чтобы проиллюстрировать желаемые результаты для различных сценариев.
* Мысли вслух: мы попросили Клода показать свой анализ в тегах <thinking>, прежде чем предоставлять окончательный результат в формате JSON.


Применяя эти методы, мы создали надежную подсказку, способную генерировать структурированные сводки для широкого спектра стенограмм звонков в службу поддержки клиентов, одновременно соответствующим образом обрабатывая крайние случаи. Этот подход можно адаптировать ко многим другим сложным сценариям подсказок, помимо суммирования вызовов.


**Важное примечание.** Хотя мы разработали сложную подсказку, которая, похоже, хорошо справляется с нашими тестовыми примерами, важно понимать, что эта подсказка еще не готова к использованию. То, что мы создали, является многообещающей отправной точкой, но требует тщательного тестирования и оценки, прежде чем его можно будет надежно использовать в реальных условиях. Наша текущая оценка теста на глаз основана на небольшом наборе примеров. Это не отражает разнообразного и часто непредсказуемого характера реальных звонков в службу поддержки клиентов. Чтобы обеспечить эффективность и надежность подсказок, нам необходимо внедрить комплексный процесс оценки, включающий количественные показатели.  Надежные оценки, основанные на данных, являются ключом к преодолению разрыва между многообещающим прототипом и надежным решением промышленного уровня.